In [1]:
import sys
sys.path.append("..")

import pandas as pd
from src.features import calcular_surpresa, calcular_ian, calcular_ice

eventos_cpi = pd.read_csv("../data/eventos.csv")
print(eventos_cpi.head())

eventos_cpi = calcular_surpresa(eventos_cpi)
print(eventos_cpi[["data", "actual", "forecast", "surpresa_zscore"]].head())

  indicador        data  actual  forecast  diferenca  surpresa_zscore
0   CPI_EUA  2026-07-14    -0.4      -0.1       -0.3        -3.022518
1   CPI_EUA  2026-06-10     0.5       0.5        0.0         0.000000
2   CPI_EUA  2026-05-12     0.6       0.6        0.0         0.000000
3   CPI_EUA  2026-04-10     0.9       1.0       -0.1        -1.007506
4   CPI_EUA  2026-03-11     0.3       0.3        0.0         0.000000
         data  actual  forecast  surpresa_zscore
0  2026-07-14    -0.4      -0.1        -3.022518
1  2026-06-10     0.5       0.5         0.000000
2  2026-05-12     0.6       0.6         0.000000
3  2026-04-10     0.9       1.0        -1.007506
4  2026-03-11     0.3       0.3         0.000000


In [2]:
eventos_cpi = calcular_ian(eventos_cpi, termos_busca=["CPI", "inflation report"], geo="US")
print(eventos_cpi[["data", "surpresa_zscore", "IAN"]].head())


        data  surpresa_zscore       IAN
0 2023-03-14         0.000000  0.211111
1 2023-04-12        -1.007506  0.311111
2 2023-05-10         0.000000  0.188889
3 2023-06-13        -1.007506  0.133333
4 2023-07-12        -1.007506  0.200000


In [3]:
eventos_cpi = calcular_ice(
    eventos_cpi,
    termos_otimistas=["abrir empresa", "comprar carro", "promoção passagens"],
    termos_pessimistas=["perder emprego", "inflação alta", "dívida"],
    geo="BR"
)
print(eventos_cpi[["data", "surpresa_zscore", "IAN", "ICE"]])

         data  surpresa_zscore       IAN       ICE
0  2023-03-14         0.000000  0.211111  0.329527
1  2023-04-12        -1.007506  0.311111  0.305865
2  2023-05-10         0.000000  0.188889  0.282809
3  2023-06-13        -1.007506  0.133333  0.253607
4  2023-07-12        -1.007506  0.200000  0.229993
5  2023-08-10         0.000000  0.211111  0.206212
6  2023-09-13         0.000000  0.122222  0.169503
7  2023-10-12         1.007506  0.088889  0.152624
8  2023-11-14        -1.007506  0.144444  0.123645
9  2023-12-12         1.007506  0.044444  0.102096
10 2024-01-11         1.007506  0.066667  0.454807
11 2024-02-13         1.007506  0.255556  0.933170
12 2024-03-12         0.000000  0.155556 -0.568122
13 2024-04-10         1.007506  0.311111 -0.129396
14 2024-05-15        -1.007506  0.244444 -0.209594
15 2024-06-12        -1.007506  0.111111 -0.371478
16 2024-07-11        -2.015012  0.177778 -0.161955
17 2024-08-14         0.000000  0.222222 -0.449987
18 2024-09-11         0.000000 

In [4]:
eventos_cpi.to_csv("../data/eventos_completo.csv", index=False)
print("Salvo com sucesso!")

Salvo com sucesso!


In [5]:
import os
print(os.listdir(r"C:\Desafio QuantAI\data"))

['eventos.csv', 'eventos_completo.csv', 'eventos_com_decisao.csv', 'eventos_com_ian.csv', 'eventos_com_retorno.csv', 'playbooks.csv', 'spy_precos.csv', 'surpresa_vs_retorno.png']


In [10]:
import pandas as pd

payroll_bruto = pd.read_csv("../data/payroll.csv", sep=";")
print(payroll_bruto.columns.tolist())
print(payroll_bruto.head())


['Release date', 'Time', 'Actual', 'Forecast', 'Previous']
         Release date   Time   Actual Forecast  Previous
0  Aug 07, 2026 (Jul)  09:30  -23.00K   85.00K    20.00K
1  Jul 02, 2026 (Jun)  09:30   57.00K  114.00K   129.00K
2  Jun 05, 2026 (May)  09:30  172.00K   85.00K   179.00K
3  May 08, 2026 (Apr)  09:30  115.00K   65.00K   185.00K
4  Apr 03, 2026 (Mar)  09:30  178.00K   65.00K  -133.00K


In [13]:
payroll_bruto["data"] = pd.to_datetime(data_limpa, format="%b %d, %Y", errors="coerce")
print(payroll_bruto[["Release date", "data"]].head(10))
print(f"Quantos viraram NaT: {payroll_bruto['data'].isna().sum()}")

          Release date       data
0   Aug 07, 2026 (Jul) 2026-08-07
1   Jul 02, 2026 (Jun) 2026-07-02
2   Jun 05, 2026 (May) 2026-06-05
3   May 08, 2026 (Apr) 2026-05-08
4   Apr 03, 2026 (Mar) 2026-04-03
5   Mar 06, 2026 (Feb) 2026-03-06
6   Feb 11, 2026 (Jan) 2026-02-11
7   Jan 09, 2026 (Dec) 2026-01-09
8   Dec 16, 2025 (Nov) 2025-12-16
10  Nov 20, 2025 (Sep) 2025-11-20
Quantos viraram NaT: 0


In [14]:
payroll_bruto = payroll_bruto[payroll_bruto["data"] >= "2023-01-01"]

payroll_bruto["indicador"] = "Payroll_EUA"
payroll_eventos = payroll_bruto[["indicador", "data", "Actual", "Forecast"]].rename(
    columns={"Actual": "actual", "Forecast": "forecast"}
)
payroll_eventos["data"] = payroll_eventos["data"].dt.strftime("%Y-%m-%d")

print(payroll_eventos)
print(f"\nTotal de linhas: {len(payroll_eventos)}")

      indicador        data  actual  forecast
0   Payroll_EUA  2026-08-07   -23.0      85.0
1   Payroll_EUA  2026-07-02    57.0     114.0
2   Payroll_EUA  2026-06-05   172.0      85.0
3   Payroll_EUA  2026-05-08   115.0      65.0
4   Payroll_EUA  2026-04-03   178.0      65.0
5   Payroll_EUA  2026-03-06   -92.0      58.0
6   Payroll_EUA  2026-02-11   130.0      66.0
7   Payroll_EUA  2026-01-09    50.0      66.0
8   Payroll_EUA  2025-12-16    64.0      51.0
10  Payroll_EUA  2025-11-20   119.0      53.0
11  Payroll_EUA  2025-09-05    22.0      75.0
12  Payroll_EUA  2025-08-01    73.0     106.0
13  Payroll_EUA  2025-07-03   147.0     111.0
14  Payroll_EUA  2025-06-06   139.0     126.0
15  Payroll_EUA  2025-05-02   177.0     138.0
16  Payroll_EUA  2025-04-04   228.0     137.0
17  Payroll_EUA  2025-03-07   151.0     159.0
18  Payroll_EUA  2025-02-07   143.0     169.0
19  Payroll_EUA  2025-01-10   256.0     164.0
20  Payroll_EUA  2024-12-06   227.0     202.0
21  Payroll_EUA  2024-11-01    12.

In [15]:
eventos_principal = pd.read_csv("../data/eventos.csv")
eventos_atualizado = pd.concat([eventos_principal, payroll_eventos], ignore_index=True)
eventos_atualizado = eventos_atualizado.drop_duplicates()
eventos_atualizado.to_csv("../data/eventos.csv", index=False)

print(eventos_atualizado["indicador"].value_counts())

indicador
Payroll_EUA    43
CPI_EUA        39
IPCA_BR        34
Selic_BR       18
Name: count, dtype: int64


In [17]:
import sys
sys.path.append("..")
from src.features import calcular_surpresa, calcular_ian, calcular_ice

payroll_completo = eventos_atualizado[eventos_atualizado["indicador"] == "Payroll_EUA"].copy()

payroll_completo = calcular_surpresa(payroll_completo)
payroll_completo = calcular_ian(payroll_completo, termos_busca=["nonfarm payrolls", "jobs report"], geo="US")
payroll_completo = calcular_ice(
    payroll_completo,
    termos_otimistas=["abrir empresa", "comprar carro", "promoção passagens"],
    termos_pessimistas=["perder emprego", "inflação alta", "dívida"],
    geo="BR"
)

print(payroll_completo[["data", "surpresa_zscore", "IAN", "ICE"]])

         data  surpresa_zscore       IAN       ICE
0  2023-01-06         0.267704  0.020619  0.466791
1  2023-02-03         3.864244  0.072165  0.478473
2  2023-03-10         1.233765  0.206186  0.391881
3  2023-04-07        -0.034918  0.082474  0.273198
4  2023-05-05         0.849668  0.020619  0.304345
5  2023-06-02         1.850647  0.041237  0.266751
6  2023-07-07        -0.186229  0.103093  0.218729
7  2023-08-04        -0.151311  0.030928  0.179730
8  2023-09-01         0.197868  0.000000  0.188734
9  2023-10-06         1.932122  0.164948  0.066134
10 2023-11-03        -0.349179  0.010309  0.058712
11 2023-12-08         0.221146  0.082474 -0.065190
12 2024-01-05         0.535407  0.020619  0.555791
13 2024-02-02         1.932122  0.041237 -0.038274
14 2024-03-08         0.896225  0.041237  0.381264
15 2024-04-05         1.059175  0.103093 -0.074885
16 2024-05-03        -0.733275  0.030928  0.273942
17 2024-06-07         1.047536  0.082474 -0.694885
18 2024-07-05         0.174589 

In [18]:
payroll_completo.to_csv("../data/eventos_payroll_completo.csv", index=False)

In [19]:
import pandas as pd

cpi = pd.read_csv("../data/eventos_completo.csv")
ipca = pd.read_csv("../data/eventos_ipca_completo.csv")
selic = pd.read_csv("../data/eventos_selic_completo.csv")
payroll = pd.read_csv("../data/eventos_payroll_completo.csv")

eventos_todos = pd.concat([cpi, ipca, selic, payroll], ignore_index=True)
eventos_todos = eventos_todos.drop_duplicates()

eventos_todos.to_csv("../data/eventos_todos_completo.csv", index=False)
print(eventos_todos["indicador"].value_counts())

indicador
Payroll_EUA    43
CPI_EUA        39
IPCA_BR        34
Selic_BR       18
Name: count, dtype: int64
